In [353]:
#!/usr/bin/env python3
# coding: utf-8
"""
This module is used to detect and associate an MGE with its bacterial MAG host.

It defines two main classes, `MAG` and `MgeMag`, along with their methods.
Starting from a `MAG` or `MgeMag` object, the module retrieves all contigs belonging to a given MAG or MGE-MAG based on binning results.

The module allows evaluation of background noise by calculating intra- and inter-MAG interaction signals, as well as interactions between MGE-MAGs and MAGs. This helps identify the host MAG for a given MGE-MAG.

Main functions and methods include:
    - Class methods:
        - `__init__()`: Initializes a MAG or MGE-MAG object.
        - `add_contig()`: Adds a contig to a MAG or MGE-MAG.
        - `has_contig()`: Checks if a MAG or MGE-MAG contains a specific contig.
        - `list_all_mags()` and `list_all_mge_mags()`: List all existing MAG and MGE-MAG objects.
    - `create_mags()` and `create_mge_mags()`: Instantiate MAG and MGE-MAG objects from input data.
    - `evaluate_experience_noise()`: Calculates intra-MAG and inter-MAG interaction signals, writes them to a file, and plots a normalized log10 boxplot.
       Also compares the distributions of intra- and inter-MAG interactions on a violin plot by quality of mags.
    - `compute_mge_mag_interactions()`: Compute MGE-MAG to MAG interaction signals with signal filtering by percentage and minimum contact threshold.
"""

# import checkv
# import metator.figures as mtf
# import metator.io as mio
from collections import Counter
from typing import Literal
import networkx as nx
import matplotlib.pyplot as plt
from metator.log import logger
import pandas as pd
import numpy as np
import seaborn as sns
import math
from typing import Union
import os
from scipy.sparse import coo_matrix


In [4]:
class Bin:
    """
        Classe de base représentant un ensemble de contigs appartenant à un même groupe (ex: MAG ou MGE-MAG).
    """

    def __init__(self, name):
        self.name: str = name
        self.info: dict = {}
        self.contigs: dict = {}
        self.completeness: float = None
        self.contamination: float = None
        self.quality: str = None
        self.intra_signal: float = None
        self.inter_signals: dict = {}
        self.inter_signal_sum : float = None
        
    def add_contig(self, contig_name, contig) -> None:
        self.contigs[contig_name] = contig

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}('{self.name}', {len(self.contigs)} contigs)"


class Mag(Bin):
    """
        Represent a MAG (Metagenome-Assembled Genome). inherit from Bin.
    """

    def __init__(self, name):
        super().__init__(name)
        self.mges: dict = {}

    def add_mge(self, mge_name, mge_mag) -> None:
        self.mges[mge_name] = mge_mag

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}('{self.name}', {len(self.contigs)} contigs, {len(self.mges)} MGEs)"


class MgeMag(Bin):
    """
        Represent a MGE-MAG (mobile genetic element). inherited from Bin.
    """

    def __init__(self, name):
        super().__init__(name)
        self.hosts: dict = {}

    def add_host(self, mag_name, mag) -> None:
        self.hosts[mag_name] = mag

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}('{self.name}', {len(self.contigs)} contigs, {len(self.hosts)} hosts)"



In [334]:
contig_data1 = pd.read_csv("/home/rsg/Documents/STAGE/DEV/host_test/test_first/contig_data_final.txt", sep="\t")
#contig_data.columns = ["Contig"] + list(contig_data.columns[1:])
bin_summary = pd.read_csv("/home/rsg/Documents/STAGE/DEV/host_test/test_first/bin_summary.txt", sep="\t")
bin_summary.columns = ["MAG"] + list(bin_summary.columns[1:])
network_data = pd.read_csv("/home/rsg/Documents/STAGE/DEV/host_test/test_first/network_0.txt", sep="\t", names=["contig1", "contig2", "signal"])
mges_bin_summary  = pd.read_csv("/home/rsg/Documents/STAGE/DEV/host_test/test_first/mges_bin_summary.tsv", sep="\t")

In [335]:
def update_final_bin_with_mge_legacy(contig_data1: pd.DataFrame, mges_bin_summary: pd.DataFrame) -> pd.DataFrame :
    """
    Updates the 'Final_bin' column of df1 by replacing 'ND' values or metator_ MAGs with the corresponding MGEs (from df2), if available.

    Args:

        contig_data (pd.DataFrame): DataFrame containing at least the columns ['Name', 'Final_bin']

        mges_bin_summary (pd.DataFrame): DataFrame containing the columns ['Contigs', 'BinName']

    Returns:

        pd.DataFrame: An updated copy of df1 with the 'Final_bin' column modified when applicable
    """

    contig_data = contig_data1.copy()
    mges_bin = mges_bin_summary.copy()

    # Construire dictionnaire contig → MGE
    mges_bin = mges_bin.assign(Contigs=mges_bin["Contigs"].str.split(",")).explode("Contigs")
    mges_bin["Contigs"] = mges_bin["Contigs"].str.strip()
    contig_to_mge = dict(zip(mges_bin["Contigs"], mges_bin["BinName"]))

    def update(row):
        name = row.get("Name")
        final_bin = row.get("Final_bin")
        mge = contig_to_mge.get(name)

        if pd.isna(final_bin):
            final_bin = "ND"  # pour que les NaN soient aussi traités comme "ND"

        if final_bin == "ND" and mge:
            return mge
        elif isinstance(final_bin, str) and final_bin.startswith("metator_") and mge:
            return mge
        else:
            return final_bin

    contig_data["Final_bin"] = contig_data.apply(update, axis=1)
    contig_data.to_csv("/home/rsg/Documents/STAGE/DEV/host_test/test_first/contig_data_df.tsv", sep="\t", index=False)

    return contig_data

def update_final_bin_with_mge_legacy2(contig_data1: pd.DataFrame, mges_bin_summary: pd.DataFrame) -> pd.DataFrame:
    """
    Updates the 'Final_bin' column of contig_data1 by replacing 'ND' values with
    the corresponding MGEs (from mges_bin_summary), if available.

    Args:
        contig_data1 (pd.DataFrame): DataFrame containing at least the columns ['Name', 'Final_bin']
        mges_bin_summary (pd.DataFrame): DataFrame with columns ['Contigs', 'BinName']

    Returns:
        pd.DataFrame: An updated copy of contig_data1 with the 'Final_bin' column modified when applicable
    """

    contig_data = contig_data1.copy()
    mges_bin = mges_bin_summary.copy()

    # Construire dictionnaire contig → MGE
    mges_bin = mges_bin.assign(Contigs=mges_bin["Contigs"].str.split(",")).explode("Contigs")
    mges_bin["Contigs"] = mges_bin["Contigs"].str.strip()
    contig_to_mge = dict(zip(mges_bin["Contigs"], mges_bin["BinName"]))

    # 🔁 Fonction mise à jour ici
    def update(row):
        name = row.get("Name")
        final_bin = row.get("Final_bin")
        mge = contig_to_mge.get(name)

        if pd.isna(final_bin) or final_bin == "ND":
            # Si le bin est manquant et un MGE est disponible, on le remplace
            return mge if mge else final_bin
        else:
            # Si déjà associé à un metator_*, on NE TOUCHE PAS
            return final_bin

    # Appliquer la mise à jour
    contig_data["Final_bin"] = contig_data.apply(update, axis=1)

    # Optionnel : sauvegarde pour debug
    contig_data.to_csv("/home/rsg/Documents/STAGE/DEV/host_test/test_first/contig_data_with_mge.tsv", sep="\t", index=False)

    return contig_data

def update_final_bin_with_mge(
    contig_data1: pd.DataFrame,
    mges_bin_summary: pd.DataFrame
) -> tuple[pd.DataFrame, list[str]]:
    """
    Updates the 'Final_bin' column of contig_data1 by replacing 'ND' values with
    the corresponding MGEs (from mges_bin_summary), if available.
    Also returns the list of contigs already assigned to a MAG and found in a MGE.

    Args:
        contig_data1 (pd.DataFrame): DataFrame containing at least ['Name', 'Final_bin']
        mges_bin_summary (pd.DataFrame): DataFrame with ['Contigs', 'BinName']

    Returns:
        tuple:
            - pd.DataFrame: Updated contig_data
            - list[str]: List of contigs assigned both to a MAG and present in MGE binning
    """

    contig_data = contig_data1.copy()
    mges_bin = mges_bin_summary.copy()

    # Construire dictionnaire contig → MGE
    mges_bin = mges_bin.assign(Contigs=mges_bin["Contigs"].str.split(",")).explode("Contigs")
    mges_bin["Contigs"] = mges_bin["Contigs"].str.strip()
    contig_to_mge = dict(zip(mges_bin["Contigs"], mges_bin["BinName"]))

    # Liste pour suivre les contigs ambigus
    ambiguous_contigs = []

    def update(row):
        name = row.get("Name")
        final_bin = row.get("Final_bin")
        mge = contig_to_mge.get(name)

        if pd.isna(final_bin) or final_bin == "ND":
            return mge if mge else final_bin
        elif isinstance(final_bin, str) and final_bin.startswith("metator_") and mge:
            ambiguous_contigs.append(name)
            return final_bin  # on ne modifie pas, mais on signale
        else:
            return final_bin

    contig_data["Final_bin"] = contig_data.apply(update, axis=1)

    # Optionnel : sauvegarde pour inspection
    #contig_data.to_csv("/home/rsg/Documents/STAGE/DEV/host_test/test_first/contig_data_with_mge.tsv", sep="\t", index=False)

    return contig_data


def create_bins(contig_data: pd.DataFrame, mges_bin_summary: pd.DataFrame) -> tuple[dict, dict]:
    """
    Creates MGE-MAGs from contig data.

    Args:
        contig_data (pd.DataFrame): DataFrame containing information about contigs.

    Returns:
        dict: A dictionary mapping MGE-MAG names to their corresponding MgeMag objects.
    """

    mags = {}
    mge_mags = {}
    # Loading data and updating the contig_data_final  to a full version with MAGs and MGEs
    logger.info("Loading data and generating the contig_data")
    contig_data = update_final_bin_with_mge(contig_data, mges_bin_summary)

    for _, row in contig_data.iterrows():
        contig_id , contig_name, bin_name = row["ID"], row["Name"], row["Final_bin"]
        #contig_id = contig_data_id 

        if bin_name.startswith("MetaTOR_MGE_"):  # Identify MGE-MAGs
            if bin_name not in mge_mags:
                mge_mags[bin_name] = MgeMag(bin_name)  # create a new MGE-MAG
            mge_mags[bin_name].add_contig(contig_name, contig_id)

        elif bin_name.startswith("metator_"):
            if bin_name not in mags:
                mags[bin_name] = Mag(bin_name)  # create a new MGE-MAG
            mags[bin_name].add_contig(contig_name, contig_id)
    logger.info(f"{len(mags)} MAGs and {len(mge_mags)} MGE-MAGs have been parsed.")

    return mags, mge_mags
    


In [336]:
mags, mge_mags  = create_bins(contig_data1, mges_bin_summary)


INFO :: Loading data and generating the contig_data
INFO :: 141 MAGs and 3152 MGE-MAGs have been parsed.


In [337]:
def map_contigs_to_mags(mags: dict) -> dict:
    """
    Creates a mapping from each contig to its corresponding MAG name.

    Iterates through a dictionary of MAG objects and associates each contig with the name of the MAG it belongs to.

    Args:
    - mags (dict): Dictionary in the form {mag_name: mag_object}, where each mag_object has a `contigs` attribute
      that returns a dictionary of contigs.

    Returns:
    - contig_to_mag (dict): Dictionary mapping each contig ID to its corresponding MAG name.
    """
    contig_to_mag = {contig: mag_name for mag_name, mag_obj in mags.items() for contig in mag_obj.contigs.values()}
    logger.info(f"{len(contig_to_mag)} contigs mapped to {len(mags)} MAGs.")

    return contig_to_mag




def map_contigs_to_mges(mge_mags: dict) -> dict:
    """
    Creates a mapping from each contig to its corresponding MAG name.

    Iterates through a dictionary of MAG objects and associates each contig with the name of the MAG it belongs to.

    Args:
    - mags (dict): Dictionary in the form {mag_name: mag_object}, where each mag_object has a `contigs` attribute
      that returns a dictionary of contigs.

    Returns:
    - contig_to_mag (dict): Dictionary mapping each contig ID to its corresponding MAG name.
    """
    contig_to_mge = {contig: mge_name for mge_name, mge_obj in mge_mags.items() for contig in mge_obj.contigs.values()}
    logger.info(f"{len(contig_to_mge)} contigs mapped to {len(mge_mags)} MGEs.")

    return contig_to_mge

def build_contig_graph(network_data: pd.DataFrame) -> nx.Graph:
    """
        Builds an undirected interaction graph from a DataFrame containing contig pairs and interaction signals.

        Each row of the DataFrame should contain two contig IDs and a signal value representing the interaction strength.
        The function constructs a NetworkX graph where each contig is a node, and each edge is weighted by the signal.

        Args:
        - network_data (pd.DataFrame): DataFrame with columns ['contig1', 'contig2', 'signal'].

        Returns:
        - G (nx.Graph): An undirected graph where edges are weighted by the interaction signal.
    """
    logger.info("Creating contigs contacts graph...")

    G = nx.Graph()
    edges = [[int(u), int(v), w] for u, v, w in zip(network_data["contig1"], network_data["contig2"], network_data["signal"])]
    G.add_weighted_edges_from(edges)
    logger.info(f"Contigs graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")
    return G


In [79]:
G = build_contig_graph(network_data)

INFO :: Creating contigs contacts graph...
INFO :: Contigs graph has 35513 nodes and 476947 edges.


In [321]:
def classify_mags(mags, bin_summary) -> dict:
    """
        Classifies MAGs based on their completeness and contamination values.

        Args:
            mags (dict): Dictionary of MAG objects.
            bin_summary (pd.DataFrame): DataFrame containing MAG quality metrics.

        Returns:
            dict: Updated dictionary of MAG objects with quality classifications.
    """
    logger.info("Classifying MAGs based on their completeness and contamination values...")

    def classify_mag(mag,) -> Union[str, None]:
        
        c, r = mag.completeness, mag.contamination
        
        if r > 1.1:
            return "Contaminated"
        elif c > 0.9 and r < 1.05:
            return "Complete"
        elif c > 0.9 and 1.05 <= r < 1.10:
            return "HQ"
        elif 0.7 < c <= 0.9 and r <= 1.10:
            return "MQ"
        if 0.5 < c <= 0.7 and r <= 1.10:
            return "LQ"
        elif c <= 0.5 and r <= 1.1:
            return "PQ"
        else:
            #if pd.isna(c) or pd.isna(r):
            return "Unknown"
        # if pd.isna(c) or pd.isna(r) or c < 0.10:
        #     return "Unknown"
        # if r > 1.1:
        #     return "Contaminated"
        # if c > 0.9 and r < 1.05:
        #     return "Complete"
        # if c > 0.9 and 1.05 <= r < 1.1:
        #     return "HQ"  # High Quality
        # if 0.7 < c <= 0.9 and r <= 1.1:
        #     return "MQ"  # Medium Quality
        # if 0.5 < c <= 0.7 and r <= 1.1:
        #     return "LQ"  # Low Quality
        # if c <= 0.5 and r <= 1.1:
        #     return "PQ"  # Poor Quality
        # else:
        #     return "Unclassified"  # si aucun critère ne matche (cas rare)
        
    for mag in mags.values():
        if mag.name in bin_summary["MAG"].values:
            mag_data = bin_summary[bin_summary["MAG"] == mag.name].iloc[0]
            mag.info = dict(mag_data)
            mag.completeness = mag_data["Completeness"]
            mag.contamination = mag_data["Redundancy"]
            mag.quality = classify_mag(mag)
        else:
            logger.warning(f"Warning: {mag.name} not found in bin summary.")

    quals = [x.quality for x in mags.values()]
    cnts = Counter(quals)
    for value, count in cnts.items():
        print(f"{value} MAGs: {count}")

    return mags

In [ ]:
def estimate_noise_v1(mags, network_data) -> pd.DataFrame:
    """
        Evaluates inter- and intra-MAG interaction signals from a contig interaction graph and
        MAG quality data, in order to detect potential noise or assembly artifacts.

        Parameters
        ----------
        mags : dict
            Dictionary where keys are MAG names and values are objects representing MAGs. Each object must have
            a `contigs` attribute that returns a dictionary of contigs.

        network_data : pd.DataFrame
            DataFrame containing interaction data with columns ['contig1', 'contig2', 'signal'].

        Returns
        ------
            Tuple (mags, background_noise) where:
            - mags (dict): MAGs enriched with intra- and inter-signal information
            - background_noise (float): Mean of inter-MAG interaction signal sums (used as baseline)

        Notes
        -----
        The function:
        - Builds a graph of contig interactions.
        - Maps each contig to its corresponding MAG.
        - Calculates average intra- and inter-MAG interaction signals.
        - Groups MAG pairs by quality (e.g., HQ-HQ, HQ-MQ, HQ-contaminated).
        - Applies log transformation and normalization to the interaction signals for downstream analysis.
    """
    logger.info("Computing the background noise of the experience.")

    # Get contigs <-> MAG mapping
    contig_to_mag = map_contigs_to_mags(mags)
    contig_to_mag_df = pd.DataFrame({"contig": list(contig_to_mag.keys()), "MAG": list(contig_to_mag.values())})

    # Merge network data with contig to MAG mapping
    mags_df = (
        network_data.merge(contig_to_mag_df, left_on="contig1", right_on="contig", how="left")
        .rename(columns={"MAG": "MAG1"})[["contig1", "contig2", "signal", "MAG1"]]
        .merge(contig_to_mag_df, left_on="contig2", right_on="contig", how="left")
        .rename(columns={"MAG": "MAG2"})[["MAG1", "MAG2", "contig1", "contig2", "signal"]]
    )

    # Define intra/inter contacts
    mags_df["type"] = np.where(mags_df["MAG1"] == mags_df["MAG2"], "intra", "inter")

    # Remove rows with NaN values
    mags_df = mags_df.dropna(subset=["MAG1", "MAG2"])
    #print(mags_df)
    ###################### INTRA-MAG INTERACTIONS COMPUTATION #################
    for _, mag in mags.items():
        intra_signals = mags_df.query("(MAG1 == @mag.name and MAG2 == @mag.name) and type == 'intra'")["signal"]
        mag.intra_signal = round(np.sum(intra_signals), 8)
        if np.isnan(mag.intra_signal) or mag.intra_signal is None:
            mag.intra_signal = 0.0
        
    ########### INTER-MAG INTERACTIONS COMPUTATION #################
    
    def get_other_mag(row, target_mag):
        if row["MAG1"] == target_mag:
            return row["MAG2"]
        else:
            return row["MAG1"]

    for _, mag in mags.items():
        subset = ((mags_df["MAG1"] == mag.name) | (mags_df["MAG2"] == mag.name)) & (mags_df["type"] == "inter")
        # if subset.sum() == 0:
        #     mag.inter_signals = 0
        #     continue
        if subset.sum() == 0:
            mag.inter_signals = [0.0] * (len(mags) - 1)
            mag.inter_signal_sum = 0.0
            continue

        else:
            #df_inters = mags_df.copy()[subset]
            df_inters = mags_df.loc[subset].copy()
            df_inters.loc[:, "mag0"] = mag.name
            df_inters.loc[:, "mag"] = df_inters.apply(get_other_mag, axis=1, target_mag=mag.name)
            scores = list(
                (
                    df_inters.groupby(["mag0", "mag"])["signal"]
                    .sum()
                    .reset_index()
                    .rename(columns={"mag0": "MAG1", "mag": "MAG2"})
                    .sort_values("MAG2")
                )["signal"]
            )
            while len(scores) < (len(mags) - 1):
                scores.append(0)
            mag.inter_signals = scores
    
        # Étape 1 : créer la matrice vide avec les MAGs triés
    mag_names = sorted(mags.keys())
    matrix = pd.DataFrame(0.0, index=mag_names, columns=mag_names)

    # Étape 2 : remplir la diagonale avec les signaux intra-MAG
    for mag_name in mag_names:
        mag = mags[mag_name]
        intra = getattr(mag, "intra_signal", 0.0)
        matrix.loc[mag_name, mag_name] = intra

    # Étape 3 : remplir les valeurs inter-MAG hors diagonale
    for _, mag in mags.items():
        mag_name = mag.name
        if not hasattr(mag, "inter_signals"):
            continue
        other_mags = [name for name in mag_names if name != mag_name]
        for other_name, signal in zip(other_mags, mag.inter_signals):
            matrix.loc[mag_name, other_name] = signal
    # Récupération des signaux intra-MAG
    intra_values = matrix.values.diagonal().tolist()
    # Calcul de la moyenne
    mean_intra = np.mean(intra_values) if intra_values else 0.0

    # 4. Moyenne des interactions inter-MAG (hors diagonale)
    # Récupérer les valeurs de la matrice sans la diagonale
    inter_values = matrix.values[~np.eye(matrix.shape[0], dtype=bool)]
    #inter_values = matrix.values[np.triu_indices_from(matrix, k=1)]
    mean_inter = np.mean(inter_values) if len(inter_values) > 0 else 0

    # Log
    logger.info(f"La moyenne des interactions intra-MAG : {mean_intra:.8f}")
    logger.info(f"La moyenne des interactions inter-MAG : {mean_inter:.8f}")
    logger.info(f"taille de inter_values  : {len(inter_values)}")
    return mags, mean_intra, mean_inter, matrix

from scipy.sparse import coo_matrix

def estimate_noise(mags, network_data):
    """
        Evaluates inter- and intra-MAG interaction signals from a contig interaction graph and
        MAG quality data, in order to detect potential noise or assembly artifacts.

        Parameters
        ----------
        mags : dict
            Dictionary where keys are MAG names and values are objects representing MAGs. Each object must have
            a `contigs` attribute that returns a dictionary of contigs.

        network_data : pd.DataFrame
            DataFrame containing interaction data with columns ['contig1', 'contig2', 'signal'].

        Returns
        ------
            Tuple (mags, background_noise) where:
            - mags (dict): MAGs enriched with intra- and inter-signal information
            - mean_intra (float): Mean of intra-MAG interaction signals
            - mean_inter (float): Mean of inter-MAG interaction signals
            - sparse_matrix (coo_matrix): Sparse full (non-symmetric, non-triangular) matrix of interactions
            - mag_names (list): List of MAG names in order of matrix indices
    """

    logger.info("Computing the background noise of the experience.")

    contig_to_mag = map_contigs_to_mags(mags)
    contig_to_mag_df = pd.DataFrame({"contig": list(contig_to_mag.keys()), "MAG": list(contig_to_mag.values())})

    mags_df = (
        network_data.merge(contig_to_mag_df, left_on="contig1", right_on="contig", how="left")
        .rename(columns={"MAG": "MAG1"})[["contig1", "contig2", "signal", "MAG1"]]
        .merge(contig_to_mag_df, left_on="contig2", right_on="contig", how="left")
        .rename(columns={"MAG": "MAG2"})[["MAG1", "MAG2", "contig1", "contig2", "signal"]]
    )

    mags_df["type"] = np.where(mags_df["MAG1"] == mags_df["MAG2"], "intra", "inter")
    mags_df = mags_df.dropna(subset=["MAG1", "MAG2"])

    for _, mag in mags.items():
        intra_signals = mags_df.query("(MAG1 == @mag.name and MAG2 == @mag.name) and type == 'intra'")["signal"]
        mag.intra_signal = round(np.sum(intra_signals), 8)
        if np.isnan(mag.intra_signal):
            mag.intra_signal = 0.0

    def get_other_mag(row, target_mag):
        return row["MAG2"] if row["MAG1"] == target_mag else row["MAG1"]

    for _, mag in mags.items():
        subset = ((mags_df["MAG1"] == mag.name) | (mags_df["MAG2"] == mag.name)) & (mags_df["type"] == "inter")
        if subset.sum() == 0:
            mag.inter_signals = [0.0] * (len(mags) - 1)
            mag.inter_signal_sum = 0.0
            continue
        df_inters = mags_df.loc[subset].copy()
        df_inters["mag0"] = mag.name
        df_inters["mag"] = df_inters.apply(get_other_mag, axis=1, target_mag=mag.name)
        scores = list(
            df_inters.groupby(["mag0", "mag"])["signal"]
            .sum()
            .reset_index()
            .rename(columns={"mag0": "MAG1", "mag": "MAG2"})
            .sort_values("MAG2")["signal"]
        )
        while len(scores) < (len(mags) - 1):
            scores.append(0)
        mag.inter_signals = scores

    # --- Build full sparse matrix (non-symmetric, non-triangular) ---
    mag_names = sorted(mags.keys())
    mag_to_idx = {name: i for i, name in enumerate(mag_names)}

    data, row, col = [], [], []

    # Diagonal (intra)
    for mag_name in mag_names:
        idx = mag_to_idx[mag_name]
        signal = getattr(mags[mag_name], "intra_signal", 0.0)
        data.append(signal)
        row.append(idx)
        col.append(idx)

    # All inter-signals as-is (non-symmetric)
    for i, mag_name in enumerate(mag_names):
        if not hasattr(mags[mag_name], "inter_signals"):
            continue
        other_names = [name for name in mag_names if name != mag_name]
        for j, other_name in enumerate(other_names):
            source_idx = mag_to_idx[mag_name]
            target_idx = mag_to_idx[other_name]
            signal = mags[mag_name].inter_signals[j]
            data.append(signal)
            row.append(source_idx)
            col.append(target_idx)

    sparse_matrix = coo_matrix((data, (row, col)), shape=(len(mag_names), len(mag_names)))

    diag_mask = np.array(row) == np.array(col)
    intra_values = np.array(data)[diag_mask]
    mean_intra = np.mean(intra_values) if intra_values.size > 0 else 0.0

    off_diag_mask = np.array(row) != np.array(col)
    inter_values = np.array(data)[off_diag_mask]
    mean_inter = np.mean(inter_values) if inter_values.size > 0 else 0.0

    logger.info(f"Moyenne intra-MAG : {mean_intra:.8f}")
    logger.info(f"Moyenne inter-MAG : {mean_inter:.8f}")
    logger.info(f"Nombre total de signaux inter-MAG : {len(inter_values)}")

    return mags, mean_intra, mean_inter, sparse_matrix, mag_names

def estimate_noise_old(mags, network_data) -> pd.DataFrame:
    logger.info("Computing the background noise of the experience.")

    # Mapping contig -> MAG
    contig_to_mag = map_contigs_to_mags(mags)
    contig_to_mag_df = pd.DataFrame({
        "contig": list(contig_to_mag.keys()),
        "MAG": list(contig_to_mag.values())
    })

    # Merge contigs & MAGs into network_data
    mags_df = (
        network_data.merge(contig_to_mag_df, left_on="contig1", right_on="contig", how="left")
        .rename(columns={"MAG": "MAG1"})[["contig1", "contig2", "signal", "MAG1"]]
        .merge(contig_to_mag_df, left_on="contig2", right_on="contig", how="left")
        .rename(columns={"MAG": "MAG2"})[["MAG1", "MAG2", "contig1", "contig2", "signal"]]
    )

    # Label intra/inter
    mags_df["type"] = np.where(mags_df["MAG1"] == mags_df["MAG2"], "intra", "inter")
    mags_df = mags_df.dropna(subset=["MAG1", "MAG2"])

    # Inter-summary records list to fill
    inter_summary_records = []

    # Compute intra-MAG signals (sum)
    for _, mag in mags.items():
        intra_signals = mags_df.query(
            "(MAG1 == @mag.name or MAG2 == @mag.name) and type == 'intra'"
        )["signal"]
        mag.intra_signal = np.sum(intra_signals)
        if np.isnan(mag.intra_signal) or mag.intra_signal is None:
            mag.intra_signal = 0.0

    # Function to get MAG2 from a row
    def get_other_mag(row, target_mag):
        return row["MAG2"] if row["MAG1"] == target_mag else row["MAG1"]

    # Compute inter-MAG signals (sum per MAG pair)
    for _, mag in mags.items():
        subset = ((mags_df["MAG1"] == mag.name) | (mags_df["MAG2"] == mag.name)) & (mags_df["type"] == "inter")

        if subset.sum() == 0:
            mag.inter_signals = [0.0] * (len(mags) - 1)
            mag.inter_signal_sum = 0.0
            continue

        df_inters = mags_df.loc[subset].copy()
        df_inters["mag0"] = mag.name
        df_inters["mag"] = df_inters.apply(get_other_mag, axis=1, target_mag=mag.name)

        df_pairs = (
            df_inters.groupby(["mag0", "mag"])["signal"]
            .sum()
            .reset_index()
            .rename(columns={"mag0": "MAG1", "mag": "MAG2"})
            .sort_values("MAG2")
        )

        # Remplir la liste d'enregistrement pour le contactmap
        for _, row in df_pairs.iterrows():
            inter_summary_records.append({
                "MAG1": row["MAG1"],
                "MAG2": row["MAG2"],
                #"sum_signal": row["signal"]
                "sum_signal": round(row["signal"], 4)
            })

        #scores = df_pairs["signal"].tolist()
        scores = [round(v, 4) for v in df_pairs["signal"].tolist()]
        while len(scores) < (len(mags) - 1):
            scores.append(0.0)
        mag.inter_signals = scores
        #mag.inter_signal_sum = sum(scores)
        mag.inter_signal_sum = round(sum(scores), 4)


    # Calcul du bruit de fond
    inter_sums = [
        mag.inter_signal_sum for mag in mags.values()
        if isinstance(mag.inter_signal_sum, (int, float))
    ]
    background_noise = np.mean(inter_sums) if inter_sums else 0.0
    logger.info(f"Bruit de fond moyen inter-MAG : {background_noise:.4f}")

    # ✅ Création du DataFrame résumé des interactions inter-MAG
    df_inter_summary = pd.DataFrame(inter_summary_records)

    return mags, background_noise, df_inter_summary


def estimate_noise2(mags, network_data):
    logger.info("Computing the background noise of the experience.")

    # Mapping contig -> MAG
    contig_to_mag = map_contigs_to_mags(mags)
    contig_to_mag_df = pd.DataFrame({
        "contig": list(contig_to_mag.keys()),
        "MAG": list(contig_to_mag.values())
    })

    # Merge contigs & MAGs into network_data
    mags_df = (
        network_data.merge(contig_to_mag_df, left_on="contig1", right_on="contig", how="left")
        .rename(columns={"MAG": "MAG1"})[["contig1", "contig2", "signal", "MAG1"]]
        .merge(contig_to_mag_df, left_on="contig2", right_on="contig", how="left")
        .rename(columns={"MAG": "MAG2"})[["MAG1", "MAG2", "contig1", "contig2", "signal"]]
    )

    

    
    # Label intra/inter
    mags_df["type"] = np.where(mags_df["MAG1"] == mags_df["MAG2"], "intra", "inter")
    # mags_df["type"] = mags_df.apply(lambda row: "intra" if row["MAG1"] == row["MAG2"] else "inter", axis=1)
    mags_df = mags_df.dropna(subset=["MAG1", "MAG2"])
    mags_df.to_csv("/home/rsg/Documents/STAGE/DEV/host_test/test_first/mag_df.tsv", sep="\t", index=False)
    

    # Liste des MAGs triés
    mag_names = sorted(mags.keys())
    n = len(mag_names)
    matrix = pd.DataFrame(np.zeros((n, n)), index=mag_names, columns=mag_names)

    # -------- Intra-MAG --------
    for _, mag in mags.items():
        intra_signals = mags_df.query("(MAG1 == @mag.name or MAG2 == @mag.name) and type == 'intra'")["signal"]
        mag.intra_signal = round(np.sum(intra_signals), 4)
        if np.isnan(mag.intra_signal):
            mag.intra_signal = 0.0
        matrix.loc[mag.name, mag.name] = mag.intra_signal

    # -------- Inter-MAG --------
    def get_other_mag(row, target_mag):
        return row["MAG2"] if row["MAG1"] == target_mag else row["MAG1"]

    for _, mag in mags.items():
        subset = ((mags_df["MAG1"] == mag.name) | (mags_df["MAG2"] == mag.name))  & (mags_df["type"] == "inter") #& (mags_df["MAG1"] != mags_df["MAG2"])
        if subset.sum() == 0:
            mag.inter_signals = [0.0] * (n - 1)
            mag.inter_signal_sum = 0.0
            continue

        df_inters = mags_df.loc[subset].copy()
        df_inters["mag0"] = mag.name
        df_inters["mag"] = df_inters.apply(get_other_mag, axis=1, target_mag=mag.name)

        df_pairs = (
            df_inters.groupby(["mag0", "mag"])["signal"]
            .sum()
            .reset_index()
            .rename(columns={"mag0": "MAG1", "mag": "MAG2"})
        )

        scores = []
        for other_mag in mag_names:
            if other_mag == mag.name:
                continue
            value = df_pairs[df_pairs["MAG2"] == other_mag]["signal"].sum()
            matrix.loc[mag.name, other_mag] = round(value, 4)
            scores.append(round(value, 4))

        mag.inter_signals = scores
        mag.inter_signal_sum = round(sum(scores), 4)

    # Calcul du bruit de fond
    inter_sums = [mag.inter_signal_sum for mag in mags.values() if isinstance(mag.inter_signal_sum, (int, float))]
    background_noise = round(np.mean(inter_sums), 4) if inter_sums else 0.0
    logger.info(f"Bruit de fond moyen inter-MAG : {background_noise:.4f}")

    return mags, background_noise, matrix


def estimate_noise3(mags, network_data):
    """
        Evaluates inter- and intra-MAG interaction signals from a contig interaction graph and
        MAG quality data, in order to detect potential noise or assembly artifacts.

        Parameters
        ----------
        mags : dict
            Dictionary where keys are MAG names and values are objects representing MAGs. Each object must have
            a `contigs` attribute that returns a dictionary of contigs.

        network_data : pd.DataFrame
            DataFrame containing interaction data with columns ['contig1', 'contig2', 'signal'].

        Returns
        ------
            Tuple (mags, mean_intra, mean_inter, sparse_matrix, mag_names) where:
            - mags (dict): MAGs enriched with intra- and inter-signal information
            - mean_intra (float): Mean intra-MAG signal
            - mean_inter (float): Mean inter-MAG signal
            - sparse_matrix (coo_matrix): Sparse upper triangular matrix of interactions
            - mag_names (list): List of MAG names in order of matrix indices
    """

    logger.info("Computing the background noise of the experience.")

    contig_to_mag = map_contigs_to_mags(mags)
    contig_to_mag_df = pd.DataFrame({"contig": list(contig_to_mag.keys()), "MAG": list(contig_to_mag.values())})

    mags_df = (
        network_data.merge(contig_to_mag_df, left_on="contig1", right_on="contig", how="left")
        .rename(columns={"MAG": "MAG1"})[["contig1", "contig2", "signal", "MAG1"]]
        .merge(contig_to_mag_df, left_on="contig2", right_on="contig", how="left")
        .rename(columns={"MAG": "MAG2"})[["MAG1", "MAG2", "contig1", "contig2", "signal"]]
    )

    mags_df["type"] = np.where(mags_df["MAG1"] == mags_df["MAG2"], "intra", "inter")
    mags_df = mags_df.dropna(subset=["MAG1", "MAG2"])

    for _, mag in mags.items():
        intra_signals = mags_df.query("(MAG1 == @mag.name and MAG2 == @mag.name) and type == 'intra'")["signal"]
        mag.intra_signal = round(np.sum(intra_signals), 8)
        if np.isnan(mag.intra_signal) or mag.intra_signal is None:
            mag.intra_signal = 0.0

    def get_other_mag(row, target_mag):
        return row["MAG2"] if row["MAG1"] == target_mag else row["MAG1"]

    for _, mag in mags.items():
        subset = ((mags_df["MAG1"] == mag.name) | (mags_df["MAG2"] == mag.name)) & (mags_df["type"] == "inter")
        if subset.sum() == 0:
            mag.inter_signals = [0.0] * (len(mags) - 1)
            mag.inter_signal_sum = 0.0
            continue
        df_inters = mags_df.loc[subset].copy()
        df_inters.loc[:, "mag0"] = mag.name
        df_inters.loc[:, "mag"] = df_inters.apply(get_other_mag, axis=1, target_mag=mag.name)
        scores = list(
            (
                df_inters.groupby(["mag0", "mag"])["signal"]
                .sum()
                .reset_index()
                .rename(columns={"mag0": "MAG1", "mag": "MAG2"})
                .sort_values("MAG2")
            )["signal"]
        )
        while len(scores) < (len(mags) - 1):
            scores.append(0)
        mag.inter_signals = scores

    # Build sparse matrix (upper triangular only)
    mag_names = sorted(mags.keys())
    mag_to_idx = {name: i for i, name in enumerate(mag_names)}

    data = []
    row = []
    col = []

    for mag_name in mag_names:
        idx = mag_to_idx[mag_name]
        intra = getattr(mags[mag_name], "intra_signal", 0.0)
        data.append(intra)
        row.append(idx)
        col.append(idx)

    for i, mag_name in enumerate(mag_names):
        if not hasattr(mags[mag_name], "inter_signals"):
            continue
        other_names = mag_names[i+1:]  # Ensure only upper triangle
        for j, other_name in enumerate(other_names):
            other_idx = mag_to_idx[other_name]
            signal = mags[mag_name].inter_signals[i + j]  # Adjust index accordingly
            data.append(signal)
            row.append(i)
            col.append(other_idx)

    sparse_matrix = coo_matrix((data, (row, col)), shape=(len(mag_names), len(mag_names)))

    diag_mask = np.array(row) == np.array(col)
    intra_values = np.array(data)[diag_mask]
    mean_intra = np.mean(intra_values) if intra_values.size > 0 else 0.0

    off_diag_mask = np.array(row) < np.array(col)  # upper triangle only
    inter_values = np.array(data)[off_diag_mask]
    mean_inter = np.mean(inter_values) if inter_values.size > 0 else 0.0

    logger.info(f"La moyenne des interactions intra-MAG : {mean_intra:.8f}")
    logger.info(f"La moyenne des interactions inter-MAG : {mean_inter:.8f}")

    return mags, mean_intra, mean_inter, sparse_matrix, mag_names


In [365]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

def plot_mag_noise(sparse_matrix, mag_names, prefix="mag_interactions", matrix_file="inter_mag_matrix.tsv"):
    """
    Visualizes the MAG interaction sparse matrix:
    - Boxplot of intra- vs inter-MAG signals (log10 transformed)
    - Heatmap of log-transformed matrix (dense version)
    - Save matrix as TSV file

    Parameters
    ----------
    sparse_matrix : scipy.sparse.coo_matrix
        Sparse interaction matrix (non-symmetric, non-triangular).
    mag_names : list of str
        List of MAG names corresponding to matrix rows/columns.
    prefix : str
        Prefix for output image filenames.
    matrix_file : str
        Path to save dense matrix as TSV.
    """
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    from scipy.sparse import coo_matrix

    logger.info("Plotting background noise from sparse matrix...")

    epsilon = 1e-6
    data = sparse_matrix.data
    row = sparse_matrix.row
    col = sparse_matrix.col

    # Identify intra/inter based on diagonal
    intra_idx = row == col
    inter_idx = row != col

    intra_values = data[intra_idx]
    inter_values = data[inter_idx]

    # Log10 transform
    intra_log = np.log10(intra_values + epsilon)
    inter_log = np.log10(inter_values + epsilon)

    # --- BOXPLOT ---
    df_box = pd.DataFrame({
        "Type": ["Intra-MAG"] * len(intra_log) + ["Inter-MAG"] * len(inter_log),
        "log10(Signal)": np.concatenate([intra_log, inter_log])
    })

    plt.figure(figsize=(4, 5))
    sns.boxplot(data=df_box, x="Type", y="log10(Signal)", palette="Set2")
    plt.title("Distribution des signaux d'interaction")
    plt.xlabel("Type d'interaction")
    plt.ylabel("log10(Signal)")
    plt.tight_layout()
    plt.savefig(f"{prefix}_boxplot.png")
    plt.close()
    logger.info(f"Boxplot sauvegardé sous : {prefix}_boxplot.png")

    # --- HEATMAP ---
    dense_matrix = sparse_matrix.toarray()
    matrix_log = np.log10(dense_matrix + epsilon)
    df_matrix = pd.DataFrame(matrix_log, index=mag_names, columns=mag_names)

    plt.figure(figsize=(20, 20))
    sns.heatmap(df_matrix, cmap="viridis", square=True, cbar_kws={"label": "log10(Signal)"})
    plt.title("Heatmap log10 des interactions MAG-MAG")
    plt.xlabel("MAG")
    plt.ylabel("MAG")
    plt.tight_layout()
    plt.savefig(f"{prefix}_heatmap.png")
    plt.close()
    logger.info(f"Heatmap sauvegardée sous : {prefix}_heatmap.png")

    # --- EXPORT TSV ---
    df_export = pd.DataFrame(dense_matrix, index=mag_names, columns=mag_names)
    df_export.to_csv(matrix_file, sep="\t")
    logger.info(f"Matrice des interactions sauvegardée sous : {matrix_file}")


In [368]:

#contig_data = pd.read_csv("/home/rsg/Documents/STAGE/DEV/host_test/test_first/contig_data_with_mge.tsv", sep="\t")

mags, mean_intra, mean_inter, sparse_matrix, mags_names = estimate_noise(mags, network_data)


INFO :: Computing the background noise of the experience.
INFO :: 88156 contigs mapped to 141 MAGs.
INFO :: Moyenne intra-MAG : 1.55756700
INFO :: Moyenne inter-MAG : 0.00135637
INFO :: Nombre total de signaux inter-MAG : 19740


In [369]:
plot_mag_noise(sparse_matrix, mags_names)

INFO :: Plotting background noise from sparse matrix...
INFO :: Boxplot sauvegardé sous : mag_interactions_boxplot.png
INFO :: Heatmap sauvegardée sous : mag_interactions_heatmap.png
INFO :: Matrice des interactions sauvegardée sous : inter_mag_matrix.tsv


In [ ]:

def compute_mge_mag_interactions(  
    network_data,
    mags,
    mge_mags,
    interaction_threshold=10.0,
    min_interacting_contigs=5,
    output_file: str = "mge_mag_interactions.tsv",
    image_file: str ="mge_mag_histogram.png",
):
    """
        Computes interactions between contigs from mgeMAGs and those from MAGs,
        and filters them based on a percentage threshold.
        Generates two histograms: one showing the total interaction signal (log scale)
        and another showing the number of MAG contigs interacting with an mgeMAG.

        Args:
        - network_data (pd.DataFrame): DataFrame with columns ['contig1', 'contig2', 'signal'].
        - mge_mags (dict): Dictionary {mgeMAG_name: MAG_object} representing the mgeMAGs.
        - mags (dict): Dictionary {MAG_name: MAG_object} representing the MAGs.
        - output_file (str): Name of the output text file.
        - image_file (str): Name of the output image file for the histograms.
        - interaction_threshold (float): Threshold (%) to validate an interaction.
        - min_interacting_contigs (int): Minimum number of MAG contigs interacting with the mgeMAG.

        Returns:
        - image_file (str): Name of the saved image file.
    """

    G = build_contig_graph(network_data)

    contig_to_mag = map_contigs_to_mags(mags)

    interaction_results = []

    for mgemag in mge_mags.values():
        interaction_signals = {}
        interacting_contigs_by_mag = {}
        for contig1 in mgemag.contigs.values():
            if contig1 in G:
                for contig2 in G.neighbors(contig1):
                    if contig2 in contig_to_mag:
                        mag_name = contig_to_mag[contig2]
                        signal = G[contig1][contig2]["weight"]
                        interaction_signals[mag_name] = interaction_signals.get(mag_name, 0) + signal
                        interacting_contigs_by_mag.setdefault(mag_name, set()).add(contig2)

        total_signal_mgemag = sum(interaction_signals.values())

        for mag_name, signal_sum in interaction_signals.items():
            percent_signal = (signal_sum / total_signal_mgemag) * 100 if total_signal_mgemag > 0 else 0
            if (percent_signal >= interaction_threshold and
                interacting_contig_count = len(interacting_contigs_by_mag[mag_name]))
                if interacting_contig_count >= min_interacting_contigs:
                    interaction_results.append((mgemag.name, mag_name, signal_sum, percent_signal, interacting_contig_count))

    with open(output_file, "w") as f:
        f.write("MGE\tMAG\tTotal Signal\tSignal rate\tInteracting Contigs\n")
        for res in interaction_results:
            mge_name, mag_name, signal_sum, percent_signal, contig_count = res
            f.write(f"{mge_name:<24}\t{mag_name:<24}\t{signal_sum:.6f}\t{percent_signal:.2f}\t{contig_count}\n")

    filtered_signals = [x[2] for x in interaction_results if x[2] > 0]
    percentage_counts = [x[3] for x in interaction_results]

    if filtered_signals and percentage_counts:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        # Histogramm of total interaction signal  (log scale)
        axes[0].hist(
            filtered_signals,
            bins=np.logspace(np.log10(min(filtered_signals)), np.log10(max(filtered_signals)), 20),
            edgecolor="black",
        )
        axes[0].set_xscale("log")
        axes[0].set_xlabel("MGE-MAG total interaction Signal  (log scale)")
        axes[0].set_ylabel("Frequences")
        axes[0].set_title(f"Signal Distribution with threshold = (\u2265 {interaction_threshold}%)")

        # Histogramm of interaction rate
        bins = range(1, math.ceil(max(percentage_counts)) + 1 + 1)  # +2 pour être équivalent à +2 dans ta ligne
        axes[1].hist(percentage_counts, bins=bins, edgecolor="black", align="left")
        axes[1].set_xlabel("Association Rate")
        axes[1].set_ylabel("Frequence")
        axes[1].set_title("Distribution of association rates of MgeMags with their MAGs.")

        plt.tight_layout()
        fig.savefig(image_file)
        plt.close(fig)

    for res in interaction_results:
        mge_name, mag_name, _, _, _ = res
        mge_mags[mge_name].add_host(mag_name=mag_name, mag=mags[mag_name])
        mags[mag_name].add_mge(mge_name=mge_name, mge_mag=mge_mags[mge_name])

    print(f"MAGs with: 0  MGEs: {len([x for x in mags.values() if len(x.mges) == 0])}")
    print(f"           1  MGEs: {len([x for x in mags.values() if len(x.mges) == 1])}")
    print(f"           2  MGEs: {len([x for x in mags.values() if len(x.mges) == 2])}")
    print(f"           3+ MGEs: {len([x for x in mags.values() if len(x.mges) >= 3])}")
    print("---")
    print(f"MGEs with: 0  hosts: {len([x for x in mge_mags.values() if len(x.hosts) == 0])}")
    print(f"           1  hosts: {len([x for x in mge_mags.values() if len(x.hosts) == 1])}")
    print(f"           2  hosts: {len([x for x in mge_mags.values() if len(x.hosts) == 2])}")
    print(f"           3  hosts: {len([x for x in mge_mags.values() if len(x.hosts) == 3])}")
    print(f"           4  hosts: {len([x for x in mge_mags.values() if len(x.hosts) == 4])}")
    print(f"           5+ hosts: {len([x for x in mge_mags.values() if len(x.hosts) >= 5])}")
    
    return mags, mge_mags



In [175]:
import pandas as pd

# Chemin d'exemple, tu devras le modifier selon ton environnement réel
file_path = "/home/rsg/Documents/STAGE/DEV/host_test/test_first/results_version1_de_uppdate/mge_mag_interactions.tsv"

# Lecture du fichier (tabulé avec en-tête)
df = pd.read_csv(file_path, sep="\t")

# Compter le nombre de MGE et de MAG uniques
n_mge = df["MGE"].nunique()
n_mag = df["MAG"].nunique()

n_mge, n_mag


(562, 60)

In [ ]:
import pandas as pd

# Chemin d'exemple, tu devras le modifier selon ton environnement réel
file_path = "/home/rsg/Documents/STAGE/DEV/host_test/test_first/results_version2_de_uppdate/mge_mag_interactions.tsv"

# Lecture du fichier (tabulé avec en-tête)
df = pd.read_csv(file_path, sep="\t")

# Chemin d'exemple, tu devras le modifier selon ton environnement réel
file_path = "/home/rsg/Documents/STAGE/DEV/host_test/test_first/results_version1_de_uppdate/mge_mag_interactions.tsv"

# Lecture du fichier (tabulé avec en-tête)
df = pd.read_csv(file_path, sep="\t")

# Compter le nombre de MGE et de MAG uniques
n_mge = df["MGE"].nunique()
n_mag = df["MAG"].nunique()

n_mge, n_mag

# Compter le nombre de MGE et de MAG uniques
n_mge = df["MGE"].nunique()
n_mag = df["MAG"].nunique()

n_mge, n_mag

(127, 29)

In [279]:
import pandas as pd

# Remplacer par le chemin réel de ton fichier
file_path = "/home/rsg/Téléchargements/data_signal_hum60.tsv"

# Lire le fichier TSV
df = pd.read_csv(file_path, sep="\t", names=["type", "name", "signal"] )

# Filtrer uniquement les lignes de type "inter" (au cas où il y en aurait d'autres)
inter_df = df[df["type"] == "intra"]

# Calculer la moyenne des signaux
mean_signal_intra = inter_df["signal"].mean()

print(f"Moyenne des signaux inter-MAG : {mean_signal_intra:.5f}")


Moyenne des signaux inter-MAG : 85.80564


In [280]:
import pandas as pd

# Remplacer par le chemin réel de ton fichier
file_path = "/home/rsg/Téléchargements/data_signal_hum60_v2.tsv"

# Lire le fichier TSV
df = pd.read_csv(file_path, sep="\t", names=["type", "name", "signal"])

# Filtrer uniquement les lignes de type "inter" (au cas où il y en aurait d'autres)
inter_df = df[df["type"] == "inter"]

# Calculer la moyenne des signaux
mean_signal_inter = inter_df["signal"].mean()

print(f"Moyenne des signaux inter-MAG : {mean_signal_inter:.5f}")

Moyenne des signaux inter-MAG : 0.12055
